In [19]:
from llama_parse import LlamaParse
import openai
from neo4j import GraphDatabase
import pandas as pd
from pathlib import Path
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Configure APIs
openai.api_key = os.getenv('OPENAI_API_KEY')
parser = LlamaParse(api_key=os.getenv('LLAMAPARSE_API_KEY'))

# Connect to Neo4j
neo4j_driver = GraphDatabase.driver(
    'bolt://localhost:7687',
    auth=(os.getenv('NEO4J_USER', 'neo4j'), os.getenv('NEO4J_PASSWORD'))
)

In [6]:
import nest_asyncio
nest_asyncio.apply()
from llama_parse import LlamaParse
import pandas as pd

# Initialize parser
parser = LlamaParse(
    api_key=os.getenv('LLAMAPARSE_API_KEY'),
    result_type="markdown",
    verbose=True
)

# Process unstructured documents
async def process_documents():
    # Define file paths
    files = [
        '../data/bom/BOM_MedicalDevice_XYZ.csv',
        '../data/eco/ECO_20250111_ComponentChange.docx',
        '../data/process/ProcessValidationReport_AssemblyLine2.pdf',
        '../data/regulatory/FDA_Submission_510k_2023K0123.pdf'
    ]
    
    # Load all documents
    documents = await parser.aload_data(files)
    
    # Extract content using get_content()
    for i, doc in enumerate(documents):
        print(f"\nDocument {i+1} Content:")
        print(doc.get_content())
        print("-" * 50)
        
        # Let's also look at metadata and relationships if any
        print(f"Metadata:")
        print(doc.metadata)
        print(f"Relationships:")
        print(doc.relationships)
        print("=" * 50)
    
    return documents

# Run document processing
documents = await process_documents()

Parsing files: 100%|██████████| 4/4 [00:13<00:00,  3.30s/it]


Document 1 Content:
|Component ID|Component Name|Device Name          |Version|Supplier Name|Notes                   |
|------------|--------------|---------------------|-------|-------------|------------------------|
|CMP-1001    |Control Unit  |GlucoMonitor Pro 2000|v1.2   |BioCore Tech |Main processing unit    |
|CMP-2002    |Adhesive Strip|GlucoMonitor Pro 2000|v1.2   |AdheSeal Inc |Disposable adhesive used|
|CMP-3003    |Sensor Array  |GlucoMonitor Pro 2000|v1.2   |SensArray LLC|Blood glucose sensor    |
|CMP-2002    |Adhesive Strip|MediPatch GlucoSensor|v3.0   |AdheSeal Inc |Adhesive for patches    |
--------------------------------------------------
Metadata:
{}
Relationships:
{}

Document 2 Content:
# Engineering Change Order #2025-0111

Date: 2025-01-11

Author: Dr. Elena Strauss, Senior Design Engineer

# Change Details:

- Component Affected: CMP-2002 (Adhesive Strip)
- Change Description
- Switching adhesive formula to improve durability and user comfort
- Justification:
-

In [5]:
print (dir(documents[0]))

['__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__class_vars__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__fields__', '__fields_set__', '__format__', '__ge__', '__get_pydantic_core_schema__', '__get_pydantic_json_schema__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__pretty__', '__private_attributes__', '__pydantic_complete__', '__pydantic_computed_fields__', '__pydantic_core_schema__', '__pydantic_custom_init__', '__pydantic_decorators__', '__pydantic_extra__', '__pydantic_fields__', '__pydantic_fields_set__', '__pydantic_generic_metadata__', '__pydantic_init_subclass__', '__pydantic_parent_namespace__', '__pydantic_post_init__', '__pydantic_private__', '__pydantic_root_model__', '__pydantic_serializer__', '__pydantic_validator__', '__reduce__', '__reduce_ex__', '__replace__', '

In [8]:
import yaml
import re
from typing import Dict, List, Tuple
from pathlib import Path

class RelationshipExtractor:
    def __init__(self, rules_file: Path):
        with open(rules_file) as f:
            self.rules = yaml.safe_load(f)
    
    def extract_entities(self, text: str) -> Dict[str, List[str]]:
        """Extract entities based on defined patterns"""
        entities = {}
        
        for entity_type, config in self.rules['entities'].items():
            pattern = config['pattern']
            matches = re.finditer(pattern, text)
            entities[entity_type] = [m.group(0) for m in matches]
            
            # Extract attributes if defined
            if 'attributes' in config:
                for attr in config['attributes']:
                    attr_matches = re.finditer(attr['pattern'], text)
                    entities[f"{entity_type}_{attr['name']}"] = [
                        m.group(1) for m in attr_matches
                    ]
        
        return entities
    
    def find_in_context(self, text: str, source: str, context_config: Dict) -> List[str]:
        """Find potential targets within specified context window"""
        # Split text into words for distance calculation
        words = text.split()
        source_idx = -1
        
        # Find source position
        for i, word in enumerate(words):
            if source in word:
                source_idx = i
                break
        
        if source_idx == -1:
            return []
        
        # Get context window
        start_idx = max(0, source_idx - context_config['source_distance'])
        end_idx = min(len(words), source_idx + context_config['source_distance'])
        context = ' '.join(words[start_idx:end_idx])
        
        # Check for keywords
        if not any(kw in context.lower() for kw in context_config['keywords']):
            return []
        
        return [context]

    def extract_relationships(self, text: str, section: str = None) -> List[Tuple]:
        """Extract relationships based on rules"""
        relationships = []
        
        # First get all entities
        entities = self.extract_entities(text)
        
        # For each relationship type
        for rel_config in self.rules['relationships']:
            source_type = rel_config['source']
            target_type = rel_config['target']
            
            # For each source entity
            for source in entities.get(source_type, []):
                # For each context configuration
                for context_config in rel_config['contexts']:
                    # Skip if section is specified but doesn't match
                    if 'sections' in context_config and section:
                        if section not in context_config['sections']:
                            continue
                    
                    # Find targets in context
                    contexts = self.find_in_context(text, source, context_config)
                    
                    # If we found valid contexts, look for targets
                    for context in contexts:
                        for target in entities.get(target_type, []):
                            if target in context:
                                relationships.append((
                                    source_type.upper(),
                                    source,
                                    rel_config['type'],
                                    target_type.upper(),
                                    target
                                ))
        
        return relationships

# Usage example:
extractor = RelationshipExtractor(Path('config/extraction_rules.yaml'))

# For each document
for doc in documents:
    print(f"\nProcessing Document:")
    print(f"Document ID: {doc.doc_id}")
    print(f"Reference ID: {doc.ref_doc_id}")
    print("=" * 50)
    
    content = doc.get_content()
    
    # Extract entities
    entities = extractor.extract_entities(content)
    print(f"Entities found:")
    for entity_type, matches in entities.items():
        print(f"{entity_type}: {matches}")
    
    # Extract relationships
    relationships = extractor.extract_relationships(content)
    print(f"\nRelationships found:")
    for rel in relationships:
        print(f"{rel[0]}({rel[1]}) -{rel[2]}-> {rel[3]}({rel[4]})")
    
    print("=" * 50)


Processing Document:
Document ID: a520bb8d-a8c9-4e3b-a9c3-f642da005387
Reference ID: None
Entities found:
component: ['CMP-1001', 'CMP-2002', 'CMP-3003', 'CMP-2002']
component_id: ['CMP-1001', 'CMP-2002', 'CMP-3003', 'CMP-2002']
component_name: []
device: ['GlucoMonitor Pro 2000', 'GlucoMonitor Pro 2000', 'GlucoMonitor Pro 2000', 'MediPatch GlucoSensor']
process: []
regulatory: []

Relationships found:

Processing Document:
Document ID: 31c004b4-08ba-46b4-9be8-dd12d2656559
Reference ID: None
Entities found:
component: ['CMP-2002']
component_id: ['CMP-2002']
component_name: ['Adhesive Strip']
device: ['GlucoMonitor Pro 2000', 'MediPatch GlucoSensor']
process: ['Assembly Line #2']
regulatory: ['2023-K0123']

Relationships found:
REGULATORY(2023-K0123) -REGULATES-> COMPONENT(CMP-2002)

Processing Document:
Document ID: d18df06f-0150-447c-8388-bb85b58bbc21
Reference ID: None
Entities found:
component: ['CMP-2002']
component_id: ['CMP-2002']
component_name: []
device: []
process: ['Assembl

In [11]:
print("Neo4j connection details")
print(f"URI: {os.getenv('NEO4J_URI')}")
print(f"User: {os.getenv('NEO4J_USER')}")
print(f"Password: {os.getenv('NEO4J_PASSWORD')}")

Neo4j connection details
URI: bolt://neo4j:7687
User: neo4j
Password: password


In [16]:
from neo4j import GraphDatabase
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

class GraphBuilder:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        
    def init_schema(self):
        """Initialize Neo4j schema with constraints and indexes"""
        with self.driver.session() as session:
            # Create constraints
            session.run("""
                CREATE CONSTRAINT IF NOT EXISTS FOR (c:Component) 
                REQUIRE c.id IS UNIQUE
            """)
            session.run("""
                CREATE CONSTRAINT IF NOT EXISTS FOR (d:Device) 
                REQUIRE d.name IS UNIQUE
            """)
            session.run("""
                CREATE CONSTRAINT IF NOT EXISTS FOR (p:Process) 
                REQUIRE p.id IS UNIQUE
            """)
            session.run("""
                CREATE CONSTRAINT IF NOT EXISTS FOR (r:Regulatory) 
                REQUIRE r.id IS UNIQUE
            """)
            
            # Check if vector index exists before creating
            result = session.run("""
                SHOW INDEXES 
                YIELD name, type
                WHERE name = 'document_embeddings' 
                AND type = 'VECTOR'
            """)
            
            if not result.single():
                # Create vector index only if it doesn't exist
                session.run("""
                    CALL db.index.vector.createNodeIndex(
                        'document_embeddings',
                        'DocumentChunk',
                        'embedding',
                        1536,
                        'cosine'
                    )
                """)
    
    def store_document_data(self, doc_id: str, entities: dict, relationships: list, content: str):
        """Store document data in Neo4j"""
        with self.driver.session() as session:
            # Store entities
            for entity_type, entity_values in entities.items():
                if entity_type.endswith('_id') or entity_type.endswith('_name'):
                    continue  # Skip attribute collections
                
                for value in entity_values:
                    session.run(f"""
                        MERGE (n:{entity_type.capitalize()} {{id: $value}})
                    """, value=value)
            
            # Store relationships
            for rel in relationships:
                source_type, source_id, rel_type, target_type, target_id = rel
                session.run(f"""
                    MATCH (source:{source_type} {{id: $source_id}})
                    MATCH (target:{target_type} {{id: $target_id}})
                    MERGE (source)-[:{rel_type}]->(target)
                """, source_id=source_id, target_id=target_id)
            
            # Store document content with embedding
            embedding_response = client.embeddings.create(
                input=content,
                model="text-embedding-ada-002"
            )
            embedding = embedding_response.data[0].embedding
            
            session.run("""
                CREATE (c:DocumentChunk {
                    doc_id: $doc_id,
                    content: $content,
                    embedding: $embedding
                })
            """, doc_id=doc_id, content=content, embedding=embedding)
    
    def close(self):
        self.driver.close()

# Usage
builder = GraphBuilder(
    uri='bolt://localhost:7687',
    user=os.getenv('NEO4J_USER'),
    password=os.getenv('NEO4J_PASSWORD')
)

# Initialize schema
builder.init_schema()

# Store data for each document
for doc in documents:
    entities = extractor.extract_entities(doc.get_content())
    relationships = extractor.extract_relationships(doc.get_content())
    builder.store_document_data(
        doc_id=doc.doc_id,
        entities=entities,
        relationships=relationships,
        content=doc.get_content()
    )

builder.close()

![Created Graph](../data/extracted_data/graph1.png)

In [25]:
class ImpactAnalyzer:
    def __init__(self, neo4j_driver, openai_client):
        self.driver = neo4j_driver
        self.openai_client = openai_client
    
    async def analyze_component_impact(self, component_id: str):
        """Analyze full impact of a component change"""
        with self.driver.session() as session:
            # Get direct relationships
            result = session.run("""
                MATCH (c:Component {id: $component_id})
                OPTIONAL MATCH (c)-[:IS_USED_IN]->(d:Device)
                OPTIONAL MATCH (c)-[:USES]->(p:Process)
                OPTIONAL MATCH (c)-[:AFFECTS]->(r:Regulatory)
                RETURN {
                    component: $component_id,
                    devices: collect(distinct CASE WHEN d IS NOT NULL THEN d.id END),
                    processes: collect(distinct CASE WHEN p IS NOT NULL THEN p.id END),
                    regulatory: collect(distinct CASE WHEN r IS NOT NULL THEN r.id END)
                } as impact
            """, component_id=component_id)
            
            direct_impact = result.single()['impact']
            
            # Get relevant document chunks using vector similarity
            embedding_response = self.openai_client.embeddings.create(
                input=f"impact analysis for component {component_id}",
                model="text-embedding-ada-002"
            )
            query_embedding = embedding_response.data[0].embedding
            
            # Get relevant document chunks using vector similarity
            relevant_docs = session.run("""
                CALL db.index.vector.queryNodes(
                    'document_embeddings',    // index name
                    $embedding,              // query vector
                    5                       // number of results
                ) YIELD node, score
                RETURN node.content as content, score
                ORDER BY score DESC
            """, embedding=query_embedding)
            
            supporting_docs = [
                {"content": doc["content"], "relevance": doc["score"]}
                for doc in relevant_docs
            ]
            
            return {
                "direct_impact": direct_impact,
                "supporting_documents": supporting_docs
            }
    
    def run_impact_query(self, query_type: str, **params):
        """Run different types of impact queries"""
        with self.driver.session() as session:
            if query_type == "component_usage":
                # Find where component is used
                result = session.run("""
                    MATCH (c:Component {id: $component_id})
                    MATCH (c)-[r]->(n)
                    RETURN type(r) as relationship, labels(n)[0] as target_type, 
                           collect(n.id) as target_ids
                """, component_id=params['component_id'])
                
                return [dict(record) for record in result]
                
            elif query_type == "process_impact":
                # Find process dependencies
                result = session.run("""
                    MATCH (p:Process {id: $process_id})
                    MATCH (p)-[r]-(n)
                    RETURN type(r) as relationship, labels(n)[0] as node_type,
                           collect(n.id) as related_ids
                """, process_id=params['process_id'])
                
                return [dict(record) for record in result]
                
            elif query_type == "regulatory_impact":
                # Find regulatory dependencies
                result = session.run("""
                    MATCH (r:Regulatory {id: $reg_id})
                    MATCH (r)-[rel]-(n)
                    RETURN type(rel) as relationship, labels(n)[0] as node_type,
                           collect(n.id) as related_ids
                """, reg_id=params['reg_id'])
                
                return [dict(record) for record in result]
            
    def check_graph_structure(self):
        """Print existing nodes and relationships in the graph"""
        with self.driver.session() as session:
            # Check node labels
            labels = session.run("""
                CALL db.labels()
                YIELD label
                RETURN collect(label) as labels
            """)
            print("Node labels:", labels.single()['labels'])

            # Check relationship types
            rels = session.run("""
                CALL db.relationshipTypes()
                YIELD relationshipType
                RETURN collect(relationshipType) as types
            """)
            print("Relationship types:", rels.single()['types'])

# Usage example
analyzer = ImpactAnalyzer(neo4j_driver, client)

# Print available relationships
analyzer.check_graph_structure()



Node labels: ['Process', 'changerequest', 'material', 'process', 'Component', 'Device', 'Regulatory', 'DocumentChunk']
Relationship types: ['AFFECTS', 'IS_USED_IN', 'USES', 'APPROVED_CHANGE']


In [26]:
# Analyze impact of CMP-2002
impact_analysis = await analyzer.analyze_component_impact("CMP-2002")
print("\nComponent Impact Analysis:")
print("Direct Impacts:", impact_analysis["direct_impact"])
print("\nSupporting Documents:")
for doc in impact_analysis["supporting_documents"]:
    print(f"\nRelevance Score: {doc['relevance']}")
    print(f"Content Preview: {doc['content'][:200]}...")

# Run specific queries
usage = analyzer.run_impact_query("component_usage", component_id="CMP-2002")
print("\nComponent Usage:")
print(usage)

CypherSyntaxError: {code: Neo.ClientError.Statement.SyntaxError} {message: Type mismatch for parameter 'embedding': expected Integer but was List<T> (line 4, column 21 (offset: 130))
"                    $embedding,              // query vector"
                     ^}